# Data Loading

In [11]:
from pathlib import Path
import zipfile
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)

## Dataset 1 *Occupational Employment and Wage Statistics Metros* 

In [12]:
zip_path = RAW / "oews_metro_may2025.zip"
oews_dir = INTERIM / "oews_metro_may2025"

if not oews_dir.exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(oews_dir)

for p in sorted(oews_dir.rglob("*")):
    if p.is_file():
        print(p.relative_to(oews_dir))

oesm25ma\BOS_M2025_dl.xlsx
oesm25ma\file_descriptions.xlsx
oesm25ma\MSA_M2025_dl.xlsx


In [13]:
oews_path = max(
    (p for p in oews_dir.rglob("*.xlsx")
     if "descriptions" not in p.name.lower()),
    key=lambda p: p.stat().st_size,
)
print("Loading:", oews_path.name)

oews = pd.read_excel(
    oews_path,
    dtype={"AREA": str, "OCC_CODE": str},
    na_values=["*", "**"],   # OEWS suppression markers
)

Loading: MSA_M2025_dl.xlsx


## Dataset 2 *Fair Market Rents*

In [15]:
fmr = pd.read_csv(RAW / "hud_fmr_fy2026.xlsx")

## Sanity Checks

In [16]:
print(oews.shape, "|", fmr.shape)
display(oews.head(3))
display(fmr.head(3))

# Data scientists only (SOC 15-2051)
ds = oews[oews["OCC_CODE"] == "15-2051"].copy()
print(f"{len(ds)} metro/area rows for data scientists")
print(ds[["A_MEDIAN"]].describe())
print("Missing median wage:", ds["A_MEDIAN"].isna().sum())

(150023, 32) | (2622, 10)


,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,21.32,29.92,45.19,24900,30980,44340,62230,93990,NaN,NaN
1,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,45.9,64.34,92.98,47790,64990,95480,133840,193390,NaN,NaN
2,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,79.7,113.51,187.68,88400,121980,165770,236100,390370,NaN,NaN


,OBJECTID,FMR_CODE,FMR_AREANAME,FMR_0BDR,FMR_1BDR,FMR_2BDR,FMR_3BDR,FMR_4BDR,Shape__Area,Shape__Length
0,1,METRO10180M10180,"Abilene, TX MSA",1010,1020,1276,1653,1987,1.005900e+10,461477.288078
1,2,METRO10380M10380,"Aguadilla, PR MSA",480,482,564,680,756,8.612798e+08,164337.142741
2,3,METRO10420M10420,"Akron, OH MSA",904,985,1268,1547,1681,4.225362e+09,291486.075708


291 metro/area rows for data scientists
        A_MEDIAN
count        282
unique       273
top        92800
freq           2
Missing median wage: 9
